# 11 · Оценка XLS-R
Только лучшая модель; test_unseen не используется для подбора порога.

In [ ]:
import subprocess
import sys
from pathlib import Path

repo=Path('/content/audio-deepfake-detector')
if not (repo/'.git').exists(): subprocess.run(['git','clone','https://github.com/Unvand/audio-deepfake-detector.git',str(repo)],check=True)
else: subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(repo/'requirements-colab-train.txt')],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers>=4.44,<5'],check=True)
sys.path.insert(0,str(repo))
commit=subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
from google.colab import drive

drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/VKR_RuASD')
prepared=base/'prepared_v1'
print('Git commit:',commit)

In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader

from audio_deepfake_detector.src.xlsr_training import (
 WaveformDataset,
 XLSRConfig,
 XLSRDetector,
 audit_manifest,
 run_epoch,
)

frame,summary=audit_manifest(prepared/'ruasd_segments_manifest.csv')
run_id='xlsr_baseline_v1'
checkpoint=base/'training_xlsr_v1/runs'/run_id/'best_model.pth'
print('Checkpoint:',checkpoint,checkpoint.is_file())

In [ ]:
RUN_EVALUATION=False
if RUN_EVALUATION:
 if not checkpoint.is_file():raise FileNotFoundError(checkpoint)
 state=torch.load(checkpoint,map_location='cpu',weights_only=False)
 cfg=XLSRConfig(**state['config']);cfg.validate()
 model=XLSRDetector.from_pretrained(cfg);model.load_state_dict(state['model'],strict=True)
 device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 model.to(device).eval()
 loader=DataLoader(WaveformDataset(frame,prepared,'test_unseen',cfg),batch_size=cfg.batch_size,shuffle=False)
 result=run_epoch(model,loader,device,cfg.threshold)
 display(pd.DataFrame({'segment':result['segment'],'source':result['source']}))
 from datetime import UTC, datetime
 from uuid import uuid4
 out=base/'training_xlsr_v1/evaluations'/run_id/('run_'+datetime.now(UTC).strftime('%Y%m%d_%H%M%S')+'_'+uuid4().hex[:8])
 out.mkdir(parents=True,exist_ok=False)
 rows=frame.loc[frame.split.eq('test_unseen')].reset_index(drop=True)
 rows['fake_score']=result['scores'];rows['predicted_label']=(rows.fake_score>=cfg.threshold).astype(int)
 rows[['segment_id','sample_id','label','fake_score','predicted_label']].to_csv(out/'segment_predictions.csv',index=False)
 source=rows.groupby('sample_id').agg(label=('label','first'),fake_score=('fake_score','mean')).reset_index()
 source['predicted_label']=(source.fake_score>=cfg.threshold).astype(int)
 source.to_csv(out/'source_predictions.csv',index=False)
 import json
 (out/'metrics.json').write_text(json.dumps({'segment':result['segment'],'source':result['source']},indent=2))
 import matplotlib.pyplot as plt
 from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, roc_auc_score
 labels=(source.label.str.lower()=='fake').astype(int)
 matrix=confusion_matrix(labels,source.predicted_label,labels=[0,1])
 fig,ax=plt.subplots(layout='constrained')
 ConfusionMatrixDisplay(matrix,display_labels=['REAL','FAKE']).plot(ax=ax)
 ax.set_title('XLS-R: test_unseen, исходные записи')
 fig.savefig(out/'confusion_matrix.png',dpi=160)
 plt.close(fig)
 print('ROC-AUC:',roc_auc_score(labels,source.fake_score))
 print('Сохранено:',out)

Внешний тест запускайте отдельно после фиксации архитектуры и настроек. Уже просмотренные 8 REAL/RVC записей нельзя использовать для выбора модели.